# Previsao Climatica - pipeline WORCAP/INPE

Roda o pipeline completo (build_dataset -> train -> predict) usando a GPU e os dados da competicao ja montados pelo Kaggle em `/kaggle/input/`.

**Antes de rodar:**
1. Settings (painel direito) -> Accelerator -> **GPU T4 x2** (ou P100).
2. Add Data -> anexa o dataset da competicao (se ainda nao estiver anexado).
3. Add Data -> Upload -> sobe a pasta do repo local (`Previsao_Climatica/`, com `src/`, `pyproject.toml` etc) como um Dataset novo -- vira `/kaggle/input/<nome-que-voce-deu>`.
4. Ajusta `CODE_INPUT` na celula abaixo pro nome exato desse dataset.

O pipeline atualizado inclui augmentation temporal para lags congelados, validação OOF do blending regional e clipping não-negativo antes do RMSE. O notebook deve ser executado de cima para baixo após atualizar o dataset de código enviado ao Kaggle.


In [ ]:
# confirma o nome exato da pasta do dataset da competicao
!ls /kaggle/input/competitions/

In [ ]:
# ajusta pro nome exato do dataset que voce subiu (Add Data -> Upload -> pasta do repo)
CODE_INPUT = "/kaggle/input/previsao-climatica-codigo"  # <-- troca pelo nome real do dataset

import os

src_dir = CODE_INPUT
if not os.path.exists(os.path.join(src_dir, "pyproject.toml")):
    # upload as vezes cria uma pasta extra aninhada (ex: .../Previsao_Climatica/Previsao_Climatica)
    subdirs = [d for d in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, d))]
    nested = [d for d in subdirs if os.path.exists(os.path.join(src_dir, d, "pyproject.toml"))]
    if nested:
        src_dir = os.path.join(src_dir, nested[0])
        print(f"detectada pasta aninhada, usando: {src_dir}")

%cd /kaggle/working
!rm -rf Previsao_Climatica
!cp -r {src_dir} Previsao_Climatica
%cd /kaggle/working/Previsao_Climatica
!ls

In [ ]:
# a imagem do Kaggle ja vem com xgboost/pandas/xarray/torch com CUDA -- so garante versoes/pacotes que podem faltar
!pip install -q pyarrow h5netcdf netcdf4 xgboost --upgrade

In [ ]:
# ajusta o nome da pasta se for diferente do que apareceu no `ls /kaggle/input/competitions/` acima
COMPETITION_INPUT = "/kaggle/input/competitions/previsao-climatica-de-precipitacao-sobre-a-america-do-sul"

!ln -sfn {COMPETITION_INPUT} data
!ls -la data/

## Índices externos causais

Baixa Niño-3.4 mensal e SOI do CPC. O pipeline usa somente o valor do último mês observado e médias móveis para evitar vazamento do mês alvo. Se a internet do Kaggle estiver indisponível, o pipeline continua usando as demais features.


In [ ]:
from pathlib import Path
from src.external_data import download_causal_indices

try:
    paths = download_causal_indices(Path("external"))
    print("índices baixados:", [str(p) for p in paths])
except Exception as exc:
    print(f"não foi possível baixar índices externos; seguindo sem eles: {exc}")


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Baseline climatologia (referencia a bater)

In [ ]:
!python -m src.baseline_climatology --split holdout

## 2. Build dataset de holdout

Além dos pares naturais (lag=1), o pipeline agora cria pseudo-previsões históricas com observação de precipitação congelada e lags 2, 3, 6, 12, 18 e 24 meses. Isso aproxima o treino da distribuição real do teste. O augmentation usa stride espacial 2 para controlar o tamanho; os pares naturais continuam em resolução cheia.


In [ ]:
# Se faltar RAM/disco, troque --spatial-stride 1 por 2.
!python -m src.build_dataset --split holdout --spatial-stride 1 --stale-spatial-stride 2 --stale-origin-step 3 --stale-lags 2,3,6,12,18,24


## Diagnóstico da climatologia no mesmo holdout

Este número deve coincidir com o baseline acima. Se divergir, há diferença entre a climatologia avaliada pelo script e a coluna usada no treinamento/blending.


In [ ]:
import pandas as pd
from sklearn.metrics import root_mean_squared_error

df_val = pd.read_parquet("processed/features_val_holdout.parquet")
print("RMSE clima_alvo no parquet:", root_mean_squared_error(df_val["tp_alvo_true"], df_val["clima_alvo"]))
print(df_val[["tp_alvo_true", "clima_alvo"]].describe())


## 3. Treino no holdout

O XGBoost agora aprende o resíduo `tp_alvo_true - clima_alvo`, que é mais estável do que a precipitação bruta. Na volta para a escala da competição, o resíduo é somado à climatologia e clipped em zero. O script também testa blend mensal e só o habilita se vencer em validação por ano.


In [ ]:
!python -m src.train --split holdout --device cuda --n-estimators 2000 --learning-rate 0.03 --max-depth 5 --min-child-weight 5 --reg-lambda 5 --reg-alpha 0.5 --target-mode residual


## 4. Treino final e submissão

Só avance se o holdout residual ficar abaixo do baseline de climatologia (1,8915). Para o treino final, usamos a mesma construção temporal do holdout e todos os dados até dezembro de 2022.


In [ ]:
!python -m src.build_dataset --split full --spatial-stride 1 --stale-spatial-stride 2 --stale-origin-step 3 --stale-lags 2,3,6,12,18,24


In [ ]:
!python -m src.train --split full --device cuda --n-estimators 2500 --learning-rate 0.03 --max-depth 5 --min-child-weight 5 --reg-lambda 5 --reg-alpha 0.5 --target-mode residual


In [ ]:
!mkdir -p /kaggle/working/submissions
!python -m src.predict --split full --output /kaggle/working/submission.csv


## 5. Submeter
`/kaggle/working/submission.csv` fica disponivel na aba **Output** do notebook -- da pra clicar em **Submit to Competition** direto dali, sem precisar baixar/subir manualmente.